In [0]:
# MAGIC Databricks Synthetic Data Generation
# MAGIC
# MAGIC Ports Fabric's `nb_generate_synthetic.ipynb` - same dbldatagen-seeded
# MAGIC approach, same seeds, same row-count targets. Exact synthetic VALUES will differ
# MAGIC from Fabric's (Spark's seeded RNG isn't guaranteed identical across different
# MAGIC clusters/partition counts), but row counts and statistical grounding (drawn from
# MAGIC the same real per-PID mean/stddev/bounds) will match - that's the actual
# MAGIC comparability guarantee here, not bit-for-bit identical synthetic rows.
# MAGIC


In [0]:
# COMMAND ----------
 
%pip install dbldatagen jmespath
 
# COMMAND ----------

In [0]:
dbutils.widgets.text("catalog", "policybench_dev")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

In [0]:
import dbldatagen as dg
from pyspark.sql import functions as F, Window
from datetime import datetime, timezone

In [0]:
def log_pipeline_run(spark, platform, layer, start_dt, end_dt):
    duration = round((end_dt - start_dt).total_seconds(), 2)
    log_row = spark.createDataFrame([{
        "platform": platform, "layer": layer,
        "start_ts": start_dt, "end_ts": end_dt, "duration_seconds": duration,
    }])
    log_row.write.format("delta").mode("append").saveAsTable("pipeline_run_log")
    print(f"[{platform}/{layer}] duration: {duration}s")

In [0]:
def generate_claim_events(spark, policyholders_df, seed: int = 42):
    with_history = policyholders_df.filter(F.col("CLM_FREQ") > 0)
 
    max_freq = with_history.agg(F.max("CLM_FREQ")).collect()[0][0]
    if not max_freq or max_freq < 1:
        max_freq = 1
 
    slots_df = (
        dg.DataGenerator(spark, name="claim_slots", rows=max_freq, partitions=2, randomSeed=seed)
        .withColumn("slot_no", "int", minValue=1, maxValue=max_freq, uniqueValues=max_freq)
        .build()
    )
 
    events = with_history.crossJoin(slots_df).filter(F.col("slot_no") <= F.col("CLM_FREQ"))
 
    events = events.withColumn("raw_weight", F.pow(F.rand(seed), 2) + 0.05)
 
    per_policy = Window.partitionBy("POLICY_ID")
    events = events.withColumn("weight_sum", F.sum("raw_weight").over(per_policy))
    events = events.withColumn(
        "claim_amount", F.round(F.col("OLDCLAIM") * F.col("raw_weight") / F.col("weight_sum"), 2)
    )
 
    events = events.withColumn("days_ago", (F.rand(seed + 1) * 1825).cast("int"))
    events = events.withColumn("claim_date", F.date_sub(F.current_date(), F.col("days_ago")))
    events = events.withColumn("claim_id", F.concat_ws("-", F.col("POLICY_ID"), F.col("slot_no")))
    events = events.withColumn("claim_type", F.lit("historical"))
 
    historical = events.select("ID", "POLICY_ID", "claim_id", "claim_date", "claim_amount", "claim_type")
 
    current = (
        policyholders_df.filter(F.col("CLAIM_FLAG") == 1)
        .withColumn("claim_id", F.concat_ws("-", F.col("POLICY_ID"), F.lit("current")))
        .withColumn("claim_date", F.date_sub(F.current_date(), (F.rand(seed + 2) * 90).cast("int")))
        .withColumn("claim_amount", F.round(F.col("CLM_AMT"), 2))
        .withColumn("claim_type", F.lit("current"))
        .select("ID", "POLICY_ID", "claim_id", "claim_date", "claim_amount", "claim_type")
    )
 
    return historical.unionByName(current)

In [0]:
def generate_synthetic_fleet(spark, real_df, n_vehicles: int = 500, readings_per_vehicle: int = 150, seed: int = 42):
    pid_stats = real_df.groupBy("PID").agg(
        F.mean("value").alias("mean_val"),
        F.stddev("value").alias("std_val"),
        F.min("value").alias("min_val"),
        F.max("value").alias("max_val"),
    )
    pid_stats = pid_stats.fillna({"std_val": 0.01})
 
    alarm_bounds = (
        real_df.groupBy("PID", "alarm_class")
        .agg(F.min("value").alias("class_min"), F.max("value").alias("class_max"))
        .withColumnRenamed("PID", "b_PID")
    )
 
    pids = [row["PID"] for row in real_df.select("PID").distinct().collect()]
    pid_df = spark.createDataFrame([(p,) for p in pids], ["PID"])
 
    devices_df = spark.range(1, n_vehicles + 1).select(
        F.concat(F.lit("SYN-"), F.lpad(F.col("id").cast("string"), 4, "0")).alias("device_id")
    )
 
    readings_df = (
        dg.DataGenerator(spark, name="readings", rows=readings_per_vehicle, partitions=4, randomSeed=seed + 1)
        .withColumn("reading_no", "int", minValue=1, maxValue=readings_per_vehicle, uniqueValues=readings_per_vehicle)
        .build()
    )
 
    events = devices_df.crossJoin(pid_df).crossJoin(readings_df)
    events = events.join(pid_stats, on="PID", how="left")
 
    events = events.withColumn("raw_value", F.col("mean_val") + F.col("std_val") * F.randn(seed + 2))
    events = events.withColumn(
        "value", F.round(F.greatest(F.col("min_val"), F.least(F.col("max_val"), F.col("raw_value"))), 3)
    )
 
    events = events.withColumn(
        "timestamp",
        (F.unix_timestamp(F.current_timestamp()) * 1000 - (F.rand(seed + 3) * 86400000).cast("long")),
    )
 
    events = events.withColumn("event_row_id", F.monotonically_increasing_id())
 
    events = events.join(
        alarm_bounds,
        on=(events["PID"] == alarm_bounds["b_PID"])
        & (events["value"] >= alarm_bounds["class_min"])
        & (events["value"] <= alarm_bounds["class_max"]),
        how="left",
    )
 
    per_event = Window.partitionBy("event_row_id").orderBy(F.desc("alarm_class"))
    events = events.withColumn("rn", F.row_number().over(per_event)).filter("rn = 1").drop("rn")
 
    events = events.fillna({"alarm_class": 0})
 
    return events.select("device_id", "timestamp", "PID", "value", "alarm_class")
 
# COMMAND ----------
 
start_dt = datetime.now(timezone.utc)
 
silver_policyholders = spark.read.table("silver_car_insurance_claim")
silver_telematics = spark.read.table("silver_telematics_events")
 
claim_events = generate_claim_events(spark, silver_policyholders, seed=42)
claim_events.write.format("delta").mode("overwrite").saveAsTable("claim_events")
 
clean_real_telematics = silver_telematics.filter(~F.col("VALUE_OUTLIER"))
synthetic_fleet = generate_synthetic_fleet(spark, clean_real_telematics, n_vehicles=500, readings_per_vehicle=150, seed=42)
synthetic_fleet.write.format("delta").mode("overwrite").saveAsTable("synthetic_telematics_fleet")
 
end_dt = datetime.now(timezone.utc)
log_pipeline_run(spark, "Databricks", "generate", start_dt, end_dt)

In [0]:
# MAGIC %md
# MAGIC ## Sanity check against the Fabric build

In [0]:
claim_events_check = spark.read.table("claim_events")
fleet_check = spark.read.table("synthetic_telematics_fleet")
 
print("claim_events rows:", claim_events_check.count())
print("synthetic_telematics_fleet rows:", fleet_check.count())